In [1]:
from openai import OpenAI
import os
import pandas as pd
from pydantic import BaseModel, Field

In [2]:
client = OpenAI(
    api_key=os.environ.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
)


In [3]:
system_prompt = """You are an expert evaluator for Large Language Model outputs.

Your task is to evaluate two model responses:
1. A BASE model response
2. A FINE-TUNED model response

Both responses answer the same user question. An EXPECTED answer is provided as the reference.

Evaluate each response independently against the EXPECTED answer and the USER QUESTION.

Evaluate using these criteria:

1. Correctness
   - Does the response provide factually correct information?
   - Does it agree with the expected answer where the expected answer contains the required facts?
   - Do not penalize a response merely because it uses different wording or structure.

2. Relevance
   - Does the response directly answer the user's question?
   - Penalize unnecessary information, irrelevant discussion, or failure to address the question.

3. Completeness
   - Does the response contain the important information present in the expected answer?
   - Penalize missing key points.

4. Helpfulness
   - Is the response clear, useful, and appropriate for the user's request?

5. Hallucination
   - Does the response introduce unsupported or incorrect information?
   - Penalize fabricated facts or claims that contradict the expected answer.

IMPORTANT:
- The EXPECTED answer is a reference, not something that must be copied verbatim.
- Equivalent wording, valid alternative explanations, and different but correct approaches should receive full credit.
- Do not judge based on writing style alone.
- Focus primarily on factual correctness and whether the user's question was properly answered.
- If the expected answer is incomplete but a model response contains additional correct information, do not penalize it.
- If the expected answer itself appears incorrect, use your own reasoning to determine correctness rather than blindly accepting it.

Give each response an overall score from 0 to 10.

Score interpretation:
9-10 = Excellent: correct, relevant, complete, and helpful
7-8 = Good: mostly correct with minor omissions or issues
5-6 = Fair: partially correct but has noticeable problems
3-4 = Poor: significant errors, omissions, or irrelevance
0-2 = Very poor: incorrect, irrelevant, or fails to answer


Return ONLY in this format:

{
  "basemodel_score": <0-10>,
  "finetuned_score": <0-10>,
  "Winner": BASE, FINE_TUNED
}"""

In [4]:
user_prompt = user_prompt = """
You are evaluating two LLM responses to the same user question.

User question:
{question}

Expected answer:
{expected_answer}

Base model response:
{base_answer}

Fine-tuned model response:
{finetuned_answer}"""

In [5]:
class Score(BaseModel):
    basemodel_score : int = Field(description='Base model output score out of 10')
    finetuned_score : int = Field(description='Fine tuned model output score out of 10')
    Winner: str = Field(description='Which model is better: BASE or FINE_TUNED')

In [6]:
baseScore=0
fineTunedScore=0
baseWin=0
fineTunedWin=0
df = pd.read_csv("TotalTestResults.csv")

values = df.values.tolist() 

In [11]:
for ques,expecAns,baseAns,fineTunedAns in values:
    response = client.chat.completions.parse(model='openai/gpt-oss-120b',messages=[{'role':'system','content':system_prompt},{'role':'user','content':user_prompt.format(
    question=ques,
    expected_answer=expecAns,
    base_answer=baseAns,
    finetuned_answer=fineTunedAns
)}],response_format=Score)
    baseScore += response.choices[0].message.parsed.basemodel_score
    fineTunedScore += response.choices[0].message.parsed.finetuned_score
    if 'base' in str(response.choices[0].message.parsed.Winner).lower():
        baseWin+=1
    else:
        fineTunedWin+=1

In [13]:
print("Base Score:", baseScore)
print("Fine-Tuned Score:", fineTunedScore)
print("Base Wins:", baseWin)
print("Fine-Tuned Wins:", fineTunedWin)
print("Base Average:", round(baseScore / 49, 2))
print("Fine-Tuned Average:", round(fineTunedScore / 49, 2))

Base Score: 221
Fine-Tuned Score: 356
Base Wins: 8
Fine-Tuned Wins: 41
Base Average: 4.51
Fine-Tuned Average: 7.27
